## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import trimesh
import numpy as np
from scripts.extractor_concurrent import ShapeExtractor
from scripts.utils import plot_ply, load_ply, build_hdf5

## 2. Single File Example

In [ ]:
# Create extractor
extractor = ShapeExtractor(smooth=1.0, seed=42)

# Extract point cloud from one file
features = extractor.extract(
    PROJECT_ROOT / "data/volumes/echinocyte3/0436.tif",
    target_points=2048,
    save_geo=False,
    save_texture=False,
)

# Plot point cloud
plot_ply(features["points"], title="Extracted Point Cloud")

In [ ]:
binary = extractor._preprocess_volume(
    PROJECT_ROOT / "data/volumes/echinocyte3/0436.tif"
)
np.save("binary.npy", binary)
mesh = extractor._extract_mesh(binary)
trimesh.exchange.export.export_mesh(mesh, "mesh.obj", file_type="obj")

## 3. Batch Process All Files

In [ ]:
# Create extractor
extractor = ShapeExtractor(smooth=1.0, seed=42)

# Batch process all files
extractor.batch_process(
    input_dir=PROJECT_ROOT / "data/volumes",
    ply_output_dir=PROJECT_ROOT / "data/pointclouds/pcl",
    metadata_path=PROJECT_ROOT / "outputs/morphology.csv",
    target_points=2048,
    save_geo=False,
    save_texture=False,
    save_binary=True,
    binary_output_dir=PROJECT_ROOT / "data/binary",
)

## 4. Build Hdf5

In [ ]:
LABEL_MAP = {
    "discocyte": 0,
    "echinocyte1": 1,
    "echinocyte2": 2,
    "echinocyte3": 3,
    "stomatocyte": 4,
    "spherocyte": 5,
    "knizocyte": 6,
}

input_path = PROJECT_ROOT / "data/pointclouds/pcl1"
output_path = PROJECT_ROOT / "outputs/rbc.h5"

build_hdf5(input_path, output_path, LABEL_MAP)